In [1]:
import pandas as pd
import numpy as np
import sklearn

print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Scikit-learn:", sklearn.__version__)


Pandas: 2.3.3
NumPy: 2.2.6
Scikit-learn: 1.7.2


In [2]:
import pandas as pd

df = pd.read_csv("Resume.csv")

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns)

print("\nFirst 5 rows:")
display(df.head())

Dataset shape: (2484, 4)

Columns:
Index(['ID', 'Resume_str', 'Resume_html', 'Category'], dtype='object')

First 5 rows:


,ID,Resume_str,Resume_html,Category
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,"<div class=""fontsize fontface vmargins hmargin...",HR
1,22323967,"HR SPECIALIST, US HR OPERATIONS ...","<div class=""fontsize fontface vmargins hmargin...",HR
2,33176873,HR DIRECTOR Summary Over 2...,"<div class=""fontsize fontface vmargins hmargin...",HR
3,27018550,HR SPECIALIST Summary Dedica...,"<div class=""fontsize fontface vmargins hmargin...",HR
4,17812897,HR MANAGER Skill Highlights ...,"<div class=""fontsize fontface vmargins hmargin...",HR


In [3]:
print("Number of unique categories:", df["Category"].nunique())

print("\nCategories:")
print(df["Category"].unique())

Number of unique categories: 24

Categories:
['HR' 'DESIGNER' 'INFORMATION-TECHNOLOGY' 'TEACHER' 'ADVOCATE'
 'BUSINESS-DEVELOPMENT' 'HEALTHCARE' 'FITNESS' 'AGRICULTURE' 'BPO' 'SALES'
 'CONSULTANT' 'DIGITAL-MEDIA' 'AUTOMOBILE' 'CHEF' 'FINANCE' 'APPAREL'
 'ENGINEERING' 'ACCOUNTANT' 'CONSTRUCTION' 'PUBLIC-RELATIONS' 'BANKING'
 'ARTS' 'AVIATION']


In [4]:
print("\nNumber of resumes in each category:")
print(df["Category"].value_counts())


Number of resumes in each category:
Category
INFORMATION-TECHNOLOGY    120
BUSINESS-DEVELOPMENT      120
ADVOCATE                  118
CHEF                      118
ENGINEERING               118
ACCOUNTANT                118
FINANCE                   118
FITNESS                   117
AVIATION                  117
SALES                     116
BANKING                   115
HEALTHCARE                115
CONSULTANT                115
CONSTRUCTION              112
PUBLIC-RELATIONS          111
HR                        110
DESIGNER                  107
ARTS                      103
TEACHER                   102
APPAREL                    97
DIGITAL-MEDIA              96
AGRICULTURE                63
AUTOMOBILE                 36
BPO                        22
Name: count, dtype: int64


In [5]:
print(df["Resume_str"].iloc[0])

         HR ADMINISTRATOR/MARKETING ASSOCIATE

HR ADMINISTRATOR       Summary     Dedicated Customer Service Manager with 15+ years of experience in Hospitality and Customer Service Management.   Respected builder and leader of customer-focused teams; strives to instill a shared, enthusiastic commitment to customer service.         Highlights         Focused on customer satisfaction  Team management  Marketing savvy  Conflict resolution techniques     Training and development  Skilled multi-tasker  Client relations specialist           Accomplishments      Missouri DOT Supervisor Training Certification  Certified by IHG in Customer Loyalty and Marketing by Segment   Hilton Worldwide General Manager Training Certification  Accomplished Trainer for cross server hospitality systems such as    Hilton OnQ  ,   Micros    Opera PMS   , Fidelio    OPERA    Reservation System (ORS) ,   Holidex    Completed courses and seminars in customer service, sales strategies, inventory control, loss preve

In [6]:
df["text_length"] = df["Resume_str"].str.len()

print(df["text_length"].describe())

count     2484.000000
mean      6295.305153
std       2769.256214
min         21.000000
25%       5160.000000
50%       5886.500000
75%       7227.250000
max      38842.000000
Name: text_length, dtype: float64


In [7]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [8]:
sample = df["Resume_str"].iloc[0]

print("BEFORE:")
print(sample[:500])

print("\nAFTER:")
print(clean_text(sample)[:500])

BEFORE:
         HR ADMINISTRATOR/MARKETING ASSOCIATE

HR ADMINISTRATOR       Summary     Dedicated Customer Service Manager with 15+ years of experience in Hospitality and Customer Service Management.   Respected builder and leader of customer-focused teams; strives to instill a shared, enthusiastic commitment to customer service.         Highlights         Focused on customer satisfaction  Team management  Marketing savvy  Conflict resolution techniques     Training and development  Skilled multi-task

AFTER:
hr administrator marketing associate hr administrator summary dedicated customer service manager with years of experience in hospitality and customer service management respected builder and leader of customer focused teams strives to instill a shared enthusiastic commitment to customer service highlights focused on customer satisfaction team management marketing savvy conflict resolution techniques training and development skilled multi tasker client relations specialist accomp

In [9]:
df["cleaned_resume"] = df["Resume_str"].apply(clean_text)

In [10]:
print(df[["Resume_str", "cleaned_resume"]].head(2))

                                          Resume_str  \
0           HR ADMINISTRATOR/MARKETING ASSOCIATE\...   
1           HR SPECIALIST, US HR OPERATIONS      ...   

                                      cleaned_resume  
0  hr administrator marketing associate hr admini...  
1  hr specialist us hr operations summary versati...  


In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words="english"
)

In [12]:
X = tfidf.fit_transform(df["cleaned_resume"])
y = df["Category"]

In [13]:
print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (2484, 5000)
y shape: (2484,)


In [14]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [15]:
print("Training samples:", X_train.shape)
print("Testing samples:", X_test.shape)
print("Training labels:", y_train.shape)
print("Testing labels:", y_test.shape)

Training samples: (1987, 5000)
Testing samples: (497, 5000)
Training labels: (1987,)
Testing labels: (497,)


In [16]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [17]:
y_pred = model.predict(X_test)

print("First 10 predictions:")
print(y_pred[:10])

First 10 predictions:
['ARTS' 'DIGITAL-MEDIA' 'CONSTRUCTION' 'HEALTHCARE' 'CONSTRUCTION' 'HR'
 'ADVOCATE' 'DIGITAL-MEDIA' 'DIGITAL-MEDIA' 'DIGITAL-MEDIA']


In [18]:
print("Actual:")
print(y_test.values[:10])

print("\nPredicted:")
print(y_pred[:10])

Actual:
['DESIGNER' 'DIGITAL-MEDIA' 'CONSTRUCTION' 'HEALTHCARE' 'CONSTRUCTION'
 'HR' 'AUTOMOBILE' 'DIGITAL-MEDIA' 'ARTS' 'DIGITAL-MEDIA']

Predicted:
['ARTS' 'DIGITAL-MEDIA' 'CONSTRUCTION' 'HEALTHCARE' 'CONSTRUCTION' 'HR'
 'ADVOCATE' 'DIGITAL-MEDIA' 'DIGITAL-MEDIA' 'DIGITAL-MEDIA']


In [19]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)
print("Accuracy (%):", accuracy * 100)

Accuracy: 0.6539235412474849
Accuracy (%): 65.3923541247485


In [20]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

                        precision    recall  f1-score   support

            ACCOUNTANT       0.67      0.83      0.74        24
              ADVOCATE       0.37      0.54      0.44        24
           AGRICULTURE       1.00      0.46      0.63        13
               APPAREL       0.67      0.21      0.32        19
                  ARTS       0.46      0.29      0.35        21
            AUTOMOBILE       0.00      0.00      0.00         7
              AVIATION       0.86      0.75      0.80        24
               BANKING       0.79      0.65      0.71        23
                   BPO       0.00      0.00      0.00         4
  BUSINESS-DEVELOPMENT       0.46      0.79      0.58        24
                  CHEF       0.81      0.71      0.76        24
          CONSTRUCTION       0.82      0.82      0.82        22
            CONSULTANT       0.44      0.17      0.25        23
              DESIGNER       0.84      0.76      0.80        21
         DIGITAL-MEDIA       0.71      

c:\Users\adeas\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\adeas\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\adeas\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(ave

In [21]:
from sklearn.svm import LinearSVC

svm_model = LinearSVC()

svm_model.fit(X_train, y_train)

,penalty,'l2'
,loss,'squared_hinge'
,dual,'auto'
,tol,0.0001
,C,1.0
,multi_class,'ovr'
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,verbose,0
,random_state,None


In [22]:
svm_pred = svm_model.predict(X_test)

svm_accuracy = accuracy_score(y_test, svm_pred)

print("Linear SVM Accuracy:", svm_accuracy)
print("Linear SVM Accuracy (%):", svm_accuracy * 100)

Linear SVM Accuracy: 0.7062374245472837
Linear SVM Accuracy (%): 70.62374245472837


In [23]:
print(classification_report(y_test, svm_pred))

                        precision    recall  f1-score   support

            ACCOUNTANT       0.71      0.83      0.77        24
              ADVOCATE       0.63      0.71      0.67        24
           AGRICULTURE       1.00      0.62      0.76        13
               APPAREL       0.60      0.32      0.41        19
                  ARTS       0.53      0.38      0.44        21
            AUTOMOBILE       1.00      0.29      0.44         7
              AVIATION       0.82      0.75      0.78        24
               BANKING       0.72      0.78      0.75        23
                   BPO       0.00      0.00      0.00         4
  BUSINESS-DEVELOPMENT       0.53      0.79      0.63        24
                  CHEF       0.85      0.71      0.77        24
          CONSTRUCTION       0.79      0.86      0.83        22
            CONSULTANT       0.64      0.30      0.41        23
              DESIGNER       0.91      0.95      0.93        21
         DIGITAL-MEDIA       0.71      

In [24]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_bigram = TfidfVectorizer(
    max_features=10000,
    stop_words="english",
    ngram_range=(1, 2)
)

X_bigram = tfidf_bigram.fit_transform(df["cleaned_resume"])

print("TF-IDF shape:", X_bigram.shape)

TF-IDF shape: (2484, 10000)


In [25]:
X_train_bigram, X_test_bigram, y_train_bigram, y_test_bigram = train_test_split(
    X_bigram,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [26]:
svm_bigram = LinearSVC()

svm_bigram.fit(X_train_bigram, y_train_bigram)

y_pred_bigram = svm_bigram.predict(X_test_bigram)

In [27]:
bigram_accuracy = accuracy_score(y_test_bigram, y_pred_bigram)

print("SVM + Bigram Accuracy:", bigram_accuracy)
print("SVM + Bigram Accuracy (%):", bigram_accuracy * 100)

SVM + Bigram Accuracy: 0.7283702213279678
SVM + Bigram Accuracy (%): 72.83702213279678


In [28]:
new_resume = """
Software engineer with experience in Python, machine learning,
data analysis, SQL, pandas, scikit-learn, and building predictive models.
Experienced in developing machine learning solutions and working with data.
"""

new_resume_cleaned = clean_text(new_resume)

new_resume_tfidf = tfidf_bigram.transform([new_resume_cleaned])

prediction = svm_bigram.predict(new_resume_tfidf)

print("Predicted Job Category:", prediction[0])

Predicted Job Category: ENGINEERING


In [29]:
# create skill dictionary for layer 2(means we will classify perticular job from broad category by using skills)
skills = [
    "python",
    "java",
    "sql",
    "machine learning",
    "deep learning",
    "artificial intelligence",
    "data science",
    "data analysis",
    "pandas",
    "numpy",
    "scikit-learn",
    "tensorflow",
    "pytorch",
    "natural language processing",
    "nlp",
    "computer vision",
    "excel",
    "power bi",
    "tableau",
    "aws",
    "azure",
    "docker",
    "kubernetes",
    "git",
    "linux",
    "html",
    "css",
    "javascript",
    "react",
    "angular",
    "node.js",
    "spring boot",
    "mongodb",
    "postgresql",
    "mysql",
    "c++",
    "c#",
    "salesforce",
    "sap",
    "project management"
]

In [30]:
# extract skills from resume
def extract_skills(text, skills):
    text = text.lower()
    
    found_skills = []
    
    for skill in skills:
        if skill in text:
            found_skills.append(skill)
    
    return found_skills

In [31]:
new_resume = """
Software engineer with experience in Python, machine learning,
data analysis, SQL, pandas, scikit-learn, and building predictive models.
Experienced in developing machine learning solutions and working with data.
"""

detected_skills = extract_skills(new_resume, skills)

print("Detected Skills:")
for skill in detected_skills:
    print("-", skill)

Detected Skills:
- python
- sql
- machine learning
- data analysis
- pandas
- scikit-learn


In [32]:
# Add role suggestion
role_skills = {
    "Software Engineer": [
        "python", "java", "sql", "javascript",
        "git", "spring boot", "c++", "c#"
    ],
    
    "Data Scientist": [
        "python", "machine learning", "data science",
        "pandas", "numpy", "scikit-learn",
        "tensorflow", "pytorch"
    ],
    
    "Data Analyst": [
        "python", "sql", "excel", "power bi",
        "tableau", "data analysis", "pandas"
    ],
    
    "ML Engineer": [
        "python", "machine learning", "deep learning",
        "tensorflow", "pytorch", "scikit-learn",
        "docker", "aws"
    ],
    
    "Web Developer": [
        "html", "css", "javascript", "react",
        "angular", "node.js"
    ]
}

In [33]:
# which role has the most matching skill
def suggest_roles(found_skills, role_skills):
    role_scores = {}

    for role, required_skills in role_skills.items():
        matches = set(found_skills).intersection(required_skills)

        score = len(matches) / len(required_skills) * 100

        role_scores[role] = {
            "score": score,
            "matched_skills": list(matches)
        }

    return sorted(
        role_scores.items(),
        key=lambda x: x[1]["score"],
        reverse=True
    )

In [34]:
role_predictions = suggest_roles(
    detected_skills,
    role_skills
)

for role, result in role_predictions:
    print(
        f"{role}: {result['score']:.1f}% "
        f"({len(result['matched_skills'])} skills matched)"
    )

Data Analyst: 57.1% (4 skills matched)
Data Scientist: 50.0% (4 skills matched)
ML Engineer: 37.5% (3 skills matched)
Software Engineer: 25.0% (2 skills matched)
Web Developer: 0.0% (0 skills matched)


In [35]:
pip install pypdf

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [36]:
from pypdf import PdfReader

In [37]:
from pypdf import PdfReader

pdf_path = "sample_resume.pdf"

reader = PdfReader(pdf_path)

pdf_text = ""

for page in reader.pages:
    text = page.extract_text()
    if text:
        pdf_text += text

print(pdf_text[:3000])

INDUSTRIAL ENGINEERING INTERN
Summary
Seeking for full time position where I can apply my technical knowledge & skills for continuous improvement. I have 26 months of experience in
Lean Manufacturing, Production Planning and Supply Chain Management. Excellent computer and analytical skills.
Highlights
Application Software: MS Office, Minitab, PowerPoint, Excel, Solid works, AutoCAD, Pro E
Database: MS Access, SAP
Programming: Java, C, C++, Visual Basic, R, Python
Accomplishments
Quality Control: Implementing of PDCA cycle for improving image quality of smartphone camera, implementing DMAIC cycle for
improving quality issues related to cosmetic damages (scratches, bent, dent, etc.) using control charts and z transformation.
Metrics & Measurements: Time study, Process Mapping, JIT, &Work Sampling of Activities.
Intro to Statistics: Data analysis using normal & exponential distribution, simple & multi linear Regression.
Project Management: Project planning, analysis, scheduling using CPM 

In [38]:
tfidf_bigram
svm_bigram

,penalty,'l2'
,loss,'squared_hinge'
,dual,'auto'
,tol,0.0001
,C,1.0
,multi_class,'ovr'
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,verbose,0
,random_state,None


In [ ]:
# Clean the extracted PDF text
pdf_cleaned = clean_text(pdf_text)

print(pdf_cleaned[:1000])

industrial engineering intern summary seeking for full time position where i can apply my technical knowledge skills for continuous improvement i have months of experience in lean manufacturing production planning and supply chain management excellent computer and analytical skills highlights application software ms office minitab powerpoint excel solid works autocad pro e database ms access sap programming java c c visual basic r python accomplishments quality control implementing of pdca cycle for improving image quality of smartphone camera implementing dmaic cycle for improving quality issues related to cosmetic damages scratches bent dent etc using control charts and z transformation metrics measurements time study process mapping jit work sampling of activities intro to statistics data analysis using normal exponential distribution simple multi linear regression project management project planning analysis scheduling using cpm and pert analysis production inventory control develo

In [ ]:
# Convert it using our trained TF-IDF
pdf_tfidf = tfidf_bigram.transform([pdf_cleaned])

In [41]:
# Predict the broad category
pdf_prediction = svm_bigram.predict(pdf_tfidf)

print("Predicted Category:", pdf_prediction[0])

Predicted Category: ENGINEERING


In [42]:
# Extract skills from the PDF
pdf_skills = extract_skills(pdf_text, skills)

print("Detected Skills:")

for skill in pdf_skills:
    print("-", skill)

Detected Skills:
- python
- java
- data analysis
- excel
- c++
- sap
- project management


In [43]:
# Suggest specific roles
pdf_role_predictions = suggest_roles(
    pdf_skills,
    role_skills
)

print("Role Suggestions:")

for role, result in pdf_role_predictions:
    print(
        f"{role}: {result['score']:.1f}% "
        f"({len(result['matched_skills'])} skills matched)"
    )

Role Suggestions:
Data Analyst: 42.9% (3 skills matched)
Software Engineer: 37.5% (3 skills matched)
Data Scientist: 12.5% (1 skills matched)
ML Engineer: 12.5% (1 skills matched)
Web Developer: 0.0% (0 skills matched)


In [ ]:
# Add these role skills 

role_skills.update({
    "Industrial Engineer": [
        "project management",
        "data analysis",
        "excel",
        "sap",
        "lean manufacturing",
        "production planning",
        "supply chain management",
        "quality control",
        "inventory management",
        "minitab",
        "autocad",
        "solidworks"
    ],

    "Mechanical Engineer": [
        "c++",
        "autocad",
        "solidworks",
        "minitab",
        "project management",
        "quality control"
    ],

    "Supply Chain Analyst": [
        "excel",
        "sap",
        "data analysis",
        "supply chain management",
        "inventory management",
        "project management"
    ]
})

In [45]:
# Expand our general skill list 
skills.extend([
    "lean manufacturing",
    "production planning",
    "supply chain management",
    "quality control",
    "inventory management",
    "minitab",
    "autocad",
    "solidworks"
])

In [46]:
# Rerun pdf skill extraction
pdf_skills = extract_skills(pdf_text, skills)

print("Detected Skills:")

for skill in pdf_skills:
    print("-", skill)

Detected Skills:
- python
- java
- data analysis
- excel
- c++
- sap
- project management
- lean manufacturing
- production planning
- supply chain management
- quality control
- inventory management
- minitab
- autocad


In [47]:
pdf_role_predictions = suggest_roles(
    pdf_skills,
    role_skills
)

print("Role Suggestions:")

for role, result in pdf_role_predictions:
    print(
        f"{role}: {result['score']:.1f}% "
        f"({len(result['matched_skills'])} skills matched)"
    )

Role Suggestions:
Supply Chain Analyst: 100.0% (6 skills matched)
Industrial Engineer: 91.7% (11 skills matched)
Mechanical Engineer: 83.3% (5 skills matched)
Data Analyst: 42.9% (3 skills matched)
Software Engineer: 37.5% (3 skills matched)
Data Scientist: 12.5% (1 skills matched)
ML Engineer: 12.5% (1 skills matched)
Web Developer: 0.0% (0 skills matched)


In [48]:
result = analyze_resume("sample_resume.pdf")

NameError: name 'analyze_resume' is not defined

In [49]:
from pypdf import PdfReader

def analyze_resume(pdf_path):
    # 1. Extract text from PDF
    reader = PdfReader(pdf_path)

    pdf_text = ""

    for page in reader.pages:
        text = page.extract_text()
        if text:
            pdf_text += text

    # 2. Clean text
    cleaned_text = clean_text(pdf_text)

    # 3. Convert text to TF-IDF
    pdf_tfidf = tfidf_bigram.transform([cleaned_text])

    # 4. Predict broad category
    predicted_category = svm_bigram.predict(pdf_tfidf)[0]

    # 5. Extract skills
    detected_skills = extract_skills(pdf_text, skills)

    # 6. Suggest roles
    role_predictions = suggest_roles(
        detected_skills,
        role_skills
    )

    # 7. Return everything
    return {
        "category": predicted_category,
        "skills": detected_skills,
        "role_suggestions": role_predictions
    }

In [50]:
result = analyze_resume("sample_resume.pdf")

In [51]:
print("Predicted Category:", result["category"])

Predicted Category: ENGINEERING


In [52]:
print("\nDetected Skills:")

for skill in result["skills"]:
    print("-", skill)


Detected Skills:
- python
- java
- data analysis
- excel
- c++
- sap
- project management
- lean manufacturing
- production planning
- supply chain management
- quality control
- inventory management
- minitab
- autocad


In [53]:
print("\nRole Suggestions:")

for role, data in result["role_suggestions"]:
    print(
        f"{role}: {data['score']:.1f}% "
        f"({len(data['matched_skills'])} skills matched)"
    )


Role Suggestions:
Supply Chain Analyst: 100.0% (6 skills matched)
Industrial Engineer: 91.7% (11 skills matched)
Mechanical Engineer: 83.3% (5 skills matched)
Data Analyst: 42.9% (3 skills matched)
Software Engineer: 37.5% (3 skills matched)
Data Scientist: 12.5% (1 skills matched)
ML Engineer: 12.5% (1 skills matched)
Web Developer: 0.0% (0 skills matched)


In [ ]:
# Save the TF-IDF vectorizer
import joblib

joblib.dump(tfidf_bigram, "tfidf_vectorizer.pkl")

['tfidf_vectorizer.pkl']

In [ ]:
# Save the SVM model
joblib.dump(svm_bigram, "resume_model.pkl")

['resume_model.pkl']